# Práctica · Clasificación con tres clases

**Estadística Descriptiva e Inferencial** · Complemento de la Clase 12

---

## Qué es esto

Una práctica corta e independiente para aplicar lo de la Clase 12 en un caso nuevo.

En la clase predijimos **sí o no** (¿sobrevivió?). Aquí hay **tres respuestas posibles**:
¿es esta flor una *setosa*, una *versicolor* o una *virginica*?

## Multiclase, no multietiqueta

Dos nombres que se confunden y significan cosas distintas:

| | Qué es | Ejemplo |
|---|---|---|
| **Multiclase** | Elegir **una** etiqueta entre varias | Esta flor es setosa **o** versicolor **o** virginica |
| **Multietiqueta** | Un caso puede tener **varias** a la vez | Un correo es «urgente» **y** «facturación» |

Hoy vemos **multiclase**, que es lo habitual.

## Los datos

El **iris de Fisher**: 150 flores, 50 de cada especie, con cuatro medidas en centímetros.
Es el dataset más usado de la historia de la estadística — lo publicó Ronald Fisher en
1936, el mismo del experimento del té de la Clase 8.

## Los cuatro bloques

| Bloque | Min | Qué haces |
|---|---|---|
| 1 | 10 | Miras los datos y ves qué separa a las especies |
| 2 | 12 | Ajustas un modelo multiclase y lees la matriz 3×3 |
| 3 | 10 | Descubres que **una especie es fácil y dos son difíciles** |
| 4 | 8 | Reduces el problema a binario y comparas |

> Tiempo total: unos 40 minutos. Se puede hacer en casa.

---
## Celda 0 · Preparación

Los datos se leen del repo del curso.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (confusion_matrix, accuracy_score,
                             classification_report, roc_auc_score)
import warnings
warnings.filterwarnings("ignore")

NAVY, BLUE, MAG, GREEN = "#0A2559", "#1A56E8", "#E6115E", "#12B886"
plt.rcParams.update({
    "figure.figsize": (7, 4.2), "figure.dpi": 110,
    "axes.grid": True, "grid.alpha": 0.25,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 11,
})

def check(nombre, obtenido, esperado, tol=1e-3):
    if obtenido is None:
        print(f"[ ] {nombre}: todavia no calculaste nada")
        return False
    ok = abs(float(obtenido) - float(esperado)) <= tol
    print(f"{'[OK]' if ok else '[X ]'} {nombre}")
    print(f"     tu resultado: {float(obtenido):.4f}   |   esperado: {float(esperado):.4f}")
    return ok

def check_bool(nombre, cond, pista=""):
    print(f"{'[OK]' if cond else '[X ]'} {nombre}")
    if not cond and pista:
        print(f"     pista: {pista}")
    return bool(cond)

URL = ("https://raw.githubusercontent.com/josefrodrim/"
       "Estad-stica-Descriptiva-E-Inferencial/main/"
       "Proyecto_final_titanic/Data/iris.csv")

try:
    iris = pd.read_csv(URL)
    print("Datos cargados desde el repo del curso.")
except Exception as e:
    print(f"No se pudo descargar ({type(e).__name__}). Sube iris.csv a Colab.")
    iris = pd.read_csv("iris.csv")

print(f"\n{len(iris)} flores, {iris.especie.nunique()} especies")
print(iris.head(4).to_string(index=False))
print()
print("Cuantas hay de cada una:")
print(iris.especie.value_counts().to_string())
print()
print("Las cuatro medidas estan en centimetros:")
print("  sepalo = la parte verde exterior de la flor")
print("  petalo = la parte de color, interior")

---
# Bloque 1 · ¿Qué separa a las especies?  ·  10 min

Regla de siempre: primero se mira.

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
medias = iris.groupby("especie").mean()

print("Media de cada medida, por especie (cm):")
print(medias.round(3).to_string())
print()
print("QUE SE VE:")
print("  El PETALO separa muchisimo:")
print(f"     largo: setosa {medias.loc['setosa','petalo_largo']:.2f}  ->  virginica {medias.loc['virginica','petalo_largo']:.2f}")
print(f"     ancho: setosa {medias.loc['setosa','petalo_ancho']:.2f}  ->  virginica {medias.loc['virginica','petalo_ancho']:.2f}")
print()
print("  El SEPALO ANCHO casi no separa, e incluso va al reves:")
print(f"     setosa {medias.loc['setosa','sepalo_ancho']:.2f}, versicolor {medias.loc['versicolor','sepalo_ancho']:.2f}, virginica {medias.loc['virginica','sepalo_ancho']:.2f}")
print()
print("Ya se puede anticipar que las medidas del petalo van a ser")
print("las variables mas utiles del modelo.")

In [ ]:
# ── VERIFICACIÓN 1.1 ─────────────────────────────────────────────────────
r = [check("pétalo largo medio de setosa", medias.loc["setosa", "petalo_largo"], 1.462),
     check("pétalo largo medio de virginica", medias.loc["virginica", "petalo_largo"], 5.552),
     check_bool("el pétalo separa mucho más que el sépalo ancho",
                (medias.petalo_largo.max() - medias.petalo_largo.min()) >
                (medias.sepalo_ancho.max() - medias.sepalo_ancho.min()))]
print()
print("1.1 OK" if all(r) else "Revisa 1.1")

### Ejercicio 1.2 — Dibújalo

Un gráfico de dispersión con las dos medidas del pétalo, coloreando por especie.

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
colores = {"setosa": GREEN, "versicolor": BLUE, "virginica": MAG}

fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))

for esp, col in colores.items():
    sub = iris[iris.especie == esp]
    ax[0].scatter(sub.petalo_largo, sub.petalo_ancho, s=55, color=col, label=esp, alpha=0.85)
    ax[1].scatter(sub.sepalo_largo, sub.sepalo_ancho, s=55, color=col, label=esp, alpha=0.85)

ax[0].set_xlabel("pétalo largo (cm)"); ax[0].set_ylabel("pétalo ancho (cm)")
ax[0].set_title("Con las medidas del PÉTALO", color=NAVY, fontweight="bold")
ax[0].legend(frameon=False)
ax[1].set_xlabel("sépalo largo (cm)"); ax[1].set_ylabel("sépalo ancho (cm)")
ax[1].set_title("Con las medidas del SÉPALO", color=NAVY, fontweight="bold")
ax[1].legend(frameon=False)
plt.tight_layout(); plt.show()

print("COMPARA LOS DOS GRAFICOS:")
print()
print("  Con el PETALO (izquierda):")
print("     setosa queda completamente separada, en su propia esquina.")
print("     versicolor y virginica se tocan un poco, pero se distinguen.")
print()
print("  Con el SEPALO (derecha):")
print("     todo se mezcla. Con estas dos medidas seria mucho mas dificil.")
print()
print("Esto ya anticipa el resultado del bloque 3: setosa sera facil,")
print("y los errores del modelo estaran entre versicolor y virginica.")

In [ ]:
# ── VERIFICACIÓN 1.2 ─────────────────────────────────────────────────────
sep_petalo = iris[iris.especie=="setosa"].petalo_largo.max() < iris[iris.especie!="setosa"].petalo_largo.min()
r = [check_bool("setosa está totalmente separada por el pétalo largo", sep_petalo,
                "ninguna setosa tiene el pétalo tan largo como la más pequeña de las otras"),
     check_bool("en cambio los sépalos se solapan",
                iris[iris.especie=="setosa"].sepalo_largo.max() >
                iris[iris.especie!="setosa"].sepalo_largo.min())]
print()
print("Bloque 1 COMPLETO" if all(r) else "Revisa 1.2")

---
# Bloque 2 · El modelo multiclase  ·  12 min

Aquí está lo bueno: **el código es exactamente el mismo de la Clase 12**.
`LogisticRegression` detecta sola que hay tres clases y se adapta.

Lo que cambia es lo que devuelve.

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
X = iris[["sepalo_largo", "sepalo_ancho", "petalo_largo", "petalo_ancho"]]
y = iris["especie"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

modelo = LogisticRegression(max_iter=1000).fit(X_train, y_train)
pred = modelo.predict(X_test)

print(f"entrenamiento = {len(X_train)} flores")
print(f"prueba        = {len(X_test)} flores")
print(f"  (stratify reparte 15 de cada especie en prueba)")
print()
print(f"EXACTITUD EN PRUEBA = {accuracy_score(y_test, pred):.4f}")
print()
print("Y ahora lo que cambia respecto a la Clase 12:")
print(f"  los coeficientes son una matriz {modelo.coef_.shape}")
print(f"  = {modelo.coef_.shape[0]} filas (una por especie) x {modelo.coef_.shape[1]} columnas (una por medida)")
print()
print("El modelo calcula la probabilidad de CADA especie y se queda con la mayor.")
print()
print("Las probabilidades de las primeras 5 flores de prueba:")
probs = modelo.predict_proba(X_test)
tabla = pd.DataFrame(probs[:5], columns=modelo.classes_).round(3)
tabla["prediccion"] = pred[:5]
tabla["real"] = y_test.values[:5]
print(tabla.to_string(index=False))
print()
print("Fijate en que las tres probabilidades de cada fila suman 1.")

In [ ]:
# ── VERIFICACIÓN 2.1 ─────────────────────────────────────────────────────
probs = modelo.predict_proba(X_test)
r = [check("flores de entrenamiento", len(X_train), 105),
     check("flores de prueba", len(X_test), 45),
     check("exactitud en prueba", accuracy_score(y_test, pred), 0.9333, tol=1e-3),
     check_bool("los coeficientes son 3x4", modelo.coef_.shape == (3, 4)),
     check_bool("las probabilidades de cada fila suman 1",
                np.allclose(probs.sum(axis=1), 1))]
print()
print("2.1 OK" if all(r) else "Revisa 2.1")

### Ejercicio 2.2 — La matriz de confusión, ahora de 3×3

Con dos clases la matriz tenía cuatro casillas. Con tres clases tiene **nueve**: la
diagonal son los aciertos y todo lo demás son errores.

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
especies = ["setosa", "versicolor", "virginica"]

cm = confusion_matrix(y_test, pred, labels=especies)

print("MATRIZ DE CONFUSION 3x3")
print("  filas = lo que era de verdad   ·   columnas = lo que predijo el modelo")
print()
tabla_cm = pd.DataFrame(cm, index=especies, columns=especies)
print(tabla_cm.to_string())
print()
aciertos = np.trace(cm)
errores = cm.sum() - aciertos
print(f"  En la DIAGONAL estan los aciertos: {aciertos} de {cm.sum()}")
print(f"  Fuera de la diagonal, los errores: {errores}")
print()
print("DONDE ESTAN LOS ERRORES:")
for i, real in enumerate(especies):
    for j, predicho in enumerate(especies):
        if i != j and cm[i, j] > 0:
            print(f"  {cm[i,j]} flor(es) que eran {real} las predijo como {predicho}")
print()
print("Y FIJATE EN LA PRIMERA FILA Y LA PRIMERA COLUMNA:")
print("  setosa no tiene NI UN error, ni en un sentido ni en el otro.")
print("  Es exactamente lo que anticipaba el grafico del bloque 1.")

In [ ]:
# ── VERIFICACIÓN 2.2 ─────────────────────────────────────────────────────
r = [check("aciertos en la diagonal", np.trace(cm), 42),
     check("errores totales", cm.sum() - np.trace(cm), 3),
     check_bool("setosa se clasifica sin ningún error",
                cm[0].sum() == cm[0, 0] and cm[:, 0].sum() == cm[0, 0]),
     check_bool("todos los errores están entre versicolor y virginica",
                cm[1, 2] + cm[2, 1] == 3)]
print()
print("Bloque 2 COMPLETO" if all(r) else "Revisa 2.2")

---
# Bloque 3 · Una exactitud global esconde tres historias  ·  10 min

El modelo acierta el 93 %. Pero ese número promedia tres situaciones muy distintas.

Con más de dos clases, **precisión y recall se calculan por clase**: cada una contra
todas las demás.

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
reporte = classification_report(y_test, pred, digits=3)

print(reporte)
print()
print("COMO SE LEE, clase por clase:")
print()
print("  setosa       precision 1.000   recall 1.000")
print("     Perfecto en las dos direcciones: ni se le escapa ninguna,")
print("     ni clasifica como setosa algo que no lo es.")
print()
print("  versicolor   precision 0.875   recall 0.933")
print("     Detecta a casi todas (recall alto), pero de las que llama")
print("     versicolor, un 12.5 % en realidad eran virginica.")
print()
print("  virginica    precision 0.929   recall 0.867")
print("     Cuando dice virginica casi siempre acierta, pero se le")
print("     escapan algunas que clasifica como versicolor.")
print()
print("=" * 66)
print("LA LECCION: la exactitud global de 0.933 esconde que una clase es")
print("trivial y las otras dos se confunden entre si.")
print("Con varias clases, SIEMPRE hay que mirar el detalle por clase.")
print("=" * 66)

In [ ]:
# ── VERIFICACIÓN 3 ──────────────────────────────────────────────────────
from sklearn.metrics import precision_score, recall_score
prec = precision_score(y_test, pred, average=None, labels=especies)
rec  = recall_score(y_test, pred, average=None, labels=especies)

r = [check("precisión de setosa", prec[0], 1.0),
     check("recall de setosa", rec[0], 1.0),
     check("precisión de versicolor", prec[1], 0.875, tol=1e-3),
     check("recall de virginica", rec[2], 0.8667, tol=1e-3),
     check_bool("setosa es la clase más fácil", prec[0] > prec[1] and prec[0] > prec[2])]
print()
print("Bloque 3 COMPLETO" if all(r) else "Revisa el bloque 3")

### ¿Por qué se confunden versicolor y virginica?

Vuelve al gráfico del bloque 1: sus nubes de puntos **se tocan**. No es un fallo del
modelo, es que esas dos especies se parecen de verdad en estas medidas.

Ningún modelo puede separar perfectamente lo que se solapa. Ese es el techo de los datos,
no del método.

---
# Bloque 4 · De vuelta a lo binario  ·  8 min

Si el problema difícil está entre **versicolor** y **virginica**, aislémoslo: quitamos
setosa y nos queda un problema binario, exactamente como el de la Clase 12.

Así podemos usar todas las herramientas de esa clase: odds ratio, AUC, umbral.

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
dos = iris[iris.especie != "setosa"].copy()
dos["es_virginica"] = (dos.especie == "virginica").astype(int)

Xb = dos[["petalo_largo", "petalo_ancho"]]
yb = dos["es_virginica"]

Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(
    Xb, yb, test_size=0.3, random_state=42, stratify=yb
)

mb = LogisticRegression(max_iter=1000).fit(Xb_tr, yb_tr)
prob_b = mb.predict_proba(Xb_te)[:, 1]
auc = roc_auc_score(yb_te, prob_b)

print(f"{len(dos)} flores (quitamos las 50 setosa)")
print(f"entrenamiento={len(Xb_tr)}  prueba={len(Xb_te)}")
print()
print(f"exactitud en prueba = {accuracy_score(yb_te, mb.predict(Xb_te)):.4f}")
print(f"AUC en prueba       = {auc:.4f}")
print()
print("LOS COEFICIENTES, como odds ratio:")
ors = np.exp(mb.coef_[0])
for nom, c, o in zip(["petalo_largo", "petalo_ancho"], mb.coef_[0], ors):
    print(f"  {nom:14} coef = {c:+.4f}   OR = {o:.2f}")
print()
print(f"  Cada centimetro mas de petalo largo multiplica por {ors[0]:.1f}")
print(f"  las odds de ser virginica, con el ancho constante.")
print()
print("EL CONTRASTE INTERESANTE:")
print(f"  AUC = {auc:.3f} es altisimo: el modelo ordena casi perfecto.")
print(f"  Pero la exactitud es {accuracy_score(yb_te, mb.predict(Xb_te)):.3f}, no 1.0.")
print()
print("  No es contradictorio. El AUC mide si el modelo ORDENA bien;")
print("  la exactitud mide si acierta con el umbral 0.5.")
print("  Hay unas pocas flores en la frontera que ningun umbral resuelve.")

In [ ]:
# ── VERIFICACIÓN 4 ──────────────────────────────────────────────────────
r = [check("flores tras quitar setosa", len(dos), 100),
     check("AUC en prueba", auc, 0.9822, tol=1e-3),
     check("exactitud binaria", accuracy_score(yb_te, mb.predict(Xb_te)), 0.9333, tol=1e-3),
     check_bool("el pétalo largo aumenta las odds de ser virginica",
                np.exp(mb.coef_[0])[0] > 1)]
print()
print("PRACTICA COMPLETA" if all(r) else "Revisa el bloque 4")

---
# Cierre

### Qué cambia y qué no, al pasar de dos clases a tres

| | Binario (Clase 12) | Multiclase (hoy) |
|---|---|---|
| El código | `LogisticRegression()` | **exactamente el mismo** |
| Coeficientes | una fila | **una fila por clase** |
| `predict_proba` | 2 columnas | **3 columnas, suman 1** |
| Matriz de confusión | 2×2 | **3×3, aciertos en la diagonal** |
| Precisión y recall | uno de cada | **uno por clase** |
| AUC | directo | necesita adaptarse (una contra el resto) |

### Los números de la práctica

| | |
|---|---|
| Exactitud multiclase | **0.9333** (42 aciertos de 45) |
| Errores | **3**, todos entre versicolor y virginica |
| setosa | precisión y recall = **1.000** |
| Coeficientes | matriz **3 × 4** |
| AUC versicolor vs virginica | **0.9822** |

### Lo que se lleva de aquí

**1 · La exactitud global esconde el detalle.** Un 93 % que en realidad son una clase
trivial y dos que se confunden. Con varias clases, mira siempre el reporte por clase.

**2 · Los errores del modelo tienen sentido en los datos.** versicolor y virginica se
solapan de verdad en el gráfico. Ningún modelo separa lo que no está separado: ese es el
techo de los datos, no del método.

**3 · El código no cambia.** Lo que cambia es qué preguntas le haces al resultado.

### Para practicar más

- Prueba el modelo **solo con las dos medidas del pétalo**. ¿Baja mucho la exactitud?
  (Pista: casi nada.)
- Repite el split con otros `random_state` y mira cuánto se mueve la exactitud. Con 45
  flores de prueba, un error de más o de menos cambia el resultado en 2 puntos.
- Aplica lo de la Clase 11: ¿están correlacionadas `petalo_largo` y `petalo_ancho`?
  (Pista: 0.963. Es multicolinealidad severa.)

---
*Estadística Descriptiva e Inferencial · Práctica complementaria de la Clase 12*